# Forward-Forward on MNIST: What Actually Moves the Needle?

A hands-on study of Hinton's **Forward-Forward (FF)** algorithm ([arXiv:2212.13345](https://arxiv.org/abs/2212.13345)).
Instead of chasing a single accuracy number, this notebook runs a **controlled-variable investigation**: it implements every
variant I tried, trains each under the *same budget*, and asks one question — **where is FF's accuracy ceiling, how big is it, and why?**

Reference implementation: github.com/mpezeshki/pytorch_forward_forward

## TL;DR — overall results (MNIST test accuracy, `[784,500,500]`, 120 epochs/layer)

| # | Variant | Test acc | Verdict |
|---|---|---|---|
| — | Teacher: backprop MLP (upper bound) | ~97.9% | reference |
| 1 | Baseline FF (goodness readout) | ~96.1% | the wall sits here |
| 2 | Input injection (concat input to every layer) | ~96.6% | speeds convergence, no higher ceiling |
| 3 | DenseNet (concat all previous layers) | ~96.0% | forward connections maxed out |
| 4 | Recurrent **top-down** (on vs off) | ~95 / ~96 | top-down contributes ~0 (even hurts) |
| 5 | Teacher coordination alignment | ~96.0% | injected coordination doesn't lift ceiling |
| 6 | Hard negatives (teacher-picked) | ~95.6% | **worse** (coverage collapse) |
| 7 | Learned **linear readout** head | ~96.6% (small) / **~97.1% (big net)** | **breaks the wall — the readout was the bottleneck** |

## The headline (scale-aware)

**The ~96% wall is largely a *readout* limitation, not the FF learning rule.** With the naive goodness-sum classifier FF caps
at ~96% no matter the architecture. But swap in a **learned linear readout** on a **big net** and it reaches **~97.1%** —
within ~1 pp of the backprop teacher. FF's features at scale are strong; the crude readout was hiding it.

## Root-cause decomposition — and how it changes with scale

`FF+goodness → FF+linear → teacher`. The gap splits into *readout* (goodness-sum vs a learned head) and *feature quality*
(FF features vs backprop features). **The split flips with net size:**

| FF net | goodness | linear | teacher | readout share | feature share |
|---|---:|---:|---:|---:|---:|
| `[784,500,500]` (small) | 96.07 | 96.59 | 97.93 | ~28% | **~72%** |
| `[784,2000,2000,2000,2000]` (big) | 95.85 | 97.12 | 97.93 | **~61%** | ~39% |

On small nets the residual gap is mostly feature quality; on big nets the **readout dominates** and FF's features come within
~0.8 pp of backprop. The `~2 pp` cost of "no global backprop" is real but **smaller than it looks with the naive readout**, and
it shrinks with scale.

| Suspect | Verdict |
|---|---|
| Architecture (width / depth / connections) | not it — all ~96 once fully trained (with naive readout) |
| Training budget | the real lever for the naive setup, but caps at ~96 |
| Inter-layer coordination (top-down / teacher-align) | injecting it doesn't move the ceiling |
| Negative data (hard / mixed) | ~96 either way; hardest-only is worse |
| **Readout (goodness-sum vs learned linear)** | **the biggest recoverable factor; grows with scale, breaks ~96 on big nets** |
| FF learning rule (local goodness, no backprop) | a real but **small** residual (~0.8 pp on big nets), shrinking with scale |

## Four methodology laws this study kept re-teaching
1. An **undertrained baseline** makes every change look like a win — fully train the baseline before comparing.
2. **Faster convergence != higher ceiling.**
3. **Higher goodness/separation != higher accuracy** (training fit != test generalization).
4. **Difference < noise = no difference** (single-seed swings ~0.3pp).

## Setup

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

In [ ]:
tf = torchvision.transforms.ToTensor()
train_set = torchvision.datasets.MNIST(".", train=True,  download=True, transform=tf)
test_set  = torchvision.datasets.MNIST(".", train=False, download=True, transform=tf)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64,  shuffle=True, drop_last=True)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False)
print("train", len(train_set), "test", len(test_set))

In [ ]:
# Label is overlaid into the first 10 pixels: FF judges "does this image match this label?"
def overlay_label(x, labels):
    x = x.clone(); x[:, :10] = 0.0
    x[torch.arange(x.shape[0], device=x.device), labels] = 1.0
    return x

# Build aligned tensors in one pass: plain image / true label / positive (img+true) / negative (img+wrong)
plain_l, lab_l, pos_l, neg_l = [], [], [], []
for images, labels in train_loader:
    xx = images.view(images.shape[0], -1)
    plain_l.append(xx); lab_l.append(labels)
    pos_l.append(overlay_label(xx, labels))
    wrong = (labels + torch.randint(1, 10, labels.shape)) % 10
    neg_l.append(overlay_label(xx, wrong))
plain_train = torch.cat(plain_l).to(device)   # plain images
label_train = torch.cat(lab_l).to(device)     # true labels
pos_train   = torch.cat(pos_l).to(device)     # positive data
neg_train   = torch.cat(neg_l).to(device)     # negative data (random wrong label)
N = plain_train.shape[0]
print("pos", pos_train.shape)

# ---- global config ----
DIMS   = [784, 500, 500]   # input + two hidden layers of 500
EPOCHS = 120               # reported numbers use 120; set to ~20 for a quick smoke test
BATCH  = 1024
RESULTS = {}               # variant name -> test accuracy

## Core FF (baseline)

Each layer is trained **locally** with a goodness objective (positive activity high, negative low), its **own optimizer**,
and no gradient flowing across layers (`.detach()` between them). Classification tries all 10 labels and picks the one with the
highest summed goodness.

In [ ]:
class FFLayer(nn.Module):
    def __init__(self, in_dim, out_dim, lr=0.03):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim); self.relu = nn.ReLU()
        self.opt = torch.optim.Adam(self.parameters(), lr=lr); self.threshold = 2.0
    def forward(self, x):
        x = x / (x.norm(dim=1, keepdim=True) + 1e-4)   # normalize: keep direction, drop length
        return self.relu(self.linear(x))
    def goodness(self, x):
        return (self.forward(x) ** 2).mean(1)
    def train_layer(self, x_pos, x_neg, epochs, batch):
        n = x_pos.shape[0]
        for _ in range(epochs):
            perm = torch.randperm(n, device=x_pos.device)
            for i in range(0, n, batch):
                idx = perm[i:i+batch]
                gp, gn = self.goodness(x_pos[idx]), self.goodness(x_neg[idx])
                loss = F.softplus(torch.cat([-(gp - self.threshold), (gn - self.threshold)])).mean()
                self.opt.zero_grad(); loss.backward(); self.opt.step()
        return self.forward(x_pos).detach(), self.forward(x_neg).detach()

class FFNet:
    def __init__(self, dims, lr=0.03):
        self.layers = [FFLayer(dims[i], dims[i+1], lr).to(device) for i in range(len(dims)-1)]
    def train(self, x_pos, x_neg, epochs=EPOCHS, batch=BATCH):
        hp, hn = x_pos, x_neg
        for layer in self.layers:
            hp, hn = layer.train_layer(hp, hn, epochs, batch)

@torch.no_grad()
def goodness_predict(net, x):
    scores = []
    for label in range(10):
        h = overlay_label(x, torch.full((x.shape[0],), label, dtype=torch.long, device=device))
        tot = 0
        for layer in net.layers:
            h = layer.forward(h); tot = tot + (h ** 2).mean(1)
        scores.append(tot.unsqueeze(1))
    return torch.cat(scores, 1).argmax(1)

@torch.no_grad()
def eval_acc(net, predict_fn=goodness_predict):
    c = t = 0
    for images, labels in test_loader:
        x = images.view(images.shape[0], -1).to(device)
        pred = predict_fn(net, x)
        c += (pred == labels.to(device)).sum().item(); t += labels.shape[0]
    return c / t * 100

### Variant 1 — Baseline FF

In [ ]:
net = FFNet(DIMS); net.train(pos_train, neg_train)
RESULTS["1. Baseline FF (goodness readout)"] = eval_acc(net)
print(f'{RESULTS["1. Baseline FF (goodness readout)"]:.2f}%')

### Variant 2 — Input injection
Concatenate the (normalized) original input to **every** layer, ResNet-style. Ablation showed the gain comes from the *image*,
not the label; and at full training it only **speeds convergence** — it does not raise the ceiling.

In [ ]:
class InjFFLayer(nn.Module):
    def __init__(self, in_dim, out_dim, lr=0.03):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim); self.relu = nn.ReLU()
        self.opt = torch.optim.Adam(self.parameters(), lr=lr); self.threshold = 2.0
    def forward(self, h, x0):
        h  = h  / (h.norm(dim=1, keepdim=True)  + 1e-4)     # normalize each source separately
        x0 = x0 / (x0.norm(dim=1, keepdim=True) + 1e-4)
        return self.relu(self.linear(torch.cat([h, x0], 1)))
    def goodness(self, h, x0):
        return (self.forward(h, x0) ** 2).mean(1)
    def train_layer(self, hp, hn, x0p, x0n, epochs, batch):
        n = hp.shape[0]
        for _ in range(epochs):
            perm = torch.randperm(n, device=hp.device)
            for i in range(0, n, batch):
                idx = perm[i:i+batch]
                gp = self.goodness(hp[idx], x0p[idx]); gn = self.goodness(hn[idx], x0n[idx])
                loss = F.softplus(torch.cat([-(gp - self.threshold), (gn - self.threshold)])).mean()
                self.opt.zero_grad(); loss.backward(); self.opt.step()
        return self.forward(hp, x0p).detach(), self.forward(hn, x0n).detach()

class InjFFNet:
    def __init__(self, dims, lr=0.03):
        # each layer input = prev output + original input -> in_dim grows by dims[0]
        self.layers = [InjFFLayer(dims[i] + dims[0], dims[i+1], lr).to(device) for i in range(len(dims)-1)]
    def train(self, x_pos, x_neg, epochs=EPOCHS, batch=BATCH):
        hp, hn = x_pos, x_neg
        for layer in self.layers:
            hp, hn = layer.train_layer(hp, hn, x_pos, x_neg, epochs, batch)   # x0 = original, same every layer

@torch.no_grad()
def inj_predict(net, x):
    scores = []
    for label in range(10):
        x0 = overlay_label(x, torch.full((x.shape[0],), label, dtype=torch.long, device=device))
        h, tot = x0, 0
        for layer in net.layers:
            h = layer.forward(h, x0); tot = tot + (h ** 2).mean(1)
        scores.append(tot.unsqueeze(1))
    return torch.cat(scores, 1).argmax(1)

net = InjFFNet(DIMS); net.train(pos_train, neg_train)
RESULTS["2. Input injection"] = eval_acc(net, inj_predict)
print(f'{RESULTS["2. Input injection"]:.2f}%')

### Variant 3 — DenseNet-style
Each layer sees **all previous layers' outputs** + the original input (each normalized separately). Deep layers stop stalling,
but test accuracy stays in the ~94-96% blob: forward connectivity is maxed out.

In [ ]:
class DenseFFLayer(nn.Module):
    def __init__(self, in_dim, out_dim, lr=0.03):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim); self.relu = nn.ReLU()
        self.opt = torch.optim.Adam(self.parameters(), lr=lr); self.threshold = 2.0
    def forward(self, sources):
        x = torch.cat([s / (s.norm(dim=1, keepdim=True) + 1e-4) for s in sources], 1)
        return self.relu(self.linear(x))
    def goodness(self, sources):
        return (self.forward(sources) ** 2).mean(1)
    def train_layer(self, sp, sn, epochs, batch):
        n = sp[0].shape[0]
        for _ in range(epochs):
            perm = torch.randperm(n, device=sp[0].device)
            for i in range(0, n, batch):
                idx = perm[i:i+batch]
                gp = self.goodness([s[idx] for s in sp]); gn = self.goodness([s[idx] for s in sn])
                loss = F.softplus(torch.cat([-(gp - self.threshold), (gn - self.threshold)])).mean()
                self.opt.zero_grad(); loss.backward(); self.opt.step()
        return self.forward(sp).detach(), self.forward(sn).detach()

class DenseFFNet:
    def __init__(self, dims, lr=0.03):
        self.layers = [DenseFFLayer(sum(dims[1:i+1]) + dims[0], dims[i+1], lr).to(device)
                       for i in range(len(dims)-1)]
    def train(self, x_pos, x_neg, epochs=EPOCHS, batch=BATCH):
        outs_p, outs_n = [], []
        for layer in self.layers:
            op, on = layer.train_layer(outs_p + [x_pos], outs_n + [x_neg], epochs, batch)
            outs_p.append(op); outs_n.append(on)

@torch.no_grad()
def dense_predict(net, x):
    scores = []
    for label in range(10):
        x0 = overlay_label(x, torch.full((x.shape[0],), label, dtype=torch.long, device=device))
        outs, tot = [], 0
        for layer in net.layers:
            h = layer.forward(outs + [x0]); tot = tot + (h ** 2).mean(1); outs.append(h)
        scores.append(tot.unsqueeze(1))
    return torch.cat(scores, 1).argmax(1)

net = DenseFFNet(DIMS); net.train(pos_train, neg_train)
RESULTS["3. DenseNet (concat all previous)"] = eval_acc(net, dense_predict)
print(f'{RESULTS["3. DenseNet (concat all previous)"]:.2f}%')

### Variant 4 — Recurrent with top-down connections

**The question.** Can we give FF the inter-layer coordination it lacks by letting *later* layers feed back into *earlier* ones —
without adding backprop?

**Why one forward pass can't do it.** In a single pass information only flows forward: when layer L computes, layer L+1 doesn't
exist yet, so there is no top-down signal to read. The trick is **time**. Run the network for `T` timesteps and let it *settle*.
At timestep `t`, each layer reads its lower neighbor's state (**bottom-up**) **and** its upper neighbor's state **from timestep
`t-1`** (**top-down**). You never read the future — only the upper layer's *previous* tick. So information climbs up over a few
ticks, then flows back down: later layers finally influence earlier ones, and **every step is still a plain forward computation
(no backprop)**.

**Two details that matter.**
1. *Synchronous update* — at each tick every layer reads the OLD states and writes into a fresh list; only after all layers are
   computed do we swap (`sp, sn = new_p, new_n`). Update in place and you collapse it back into an ordinary feed-forward pass.
2. *Normalize each source separately* — length is goodness, so we keep only each neighbor's direction.

**The ablation.** `use_topdown = True` vs `False` on the same architecture and budget. This isolates the top-down contribution
from everything else the recurrent scaffold does (e.g. more updates).

**Result.** Top-down contributes **~0 — here it even hurts** (ON 95.3% < OFF 96.4%). Any edge over the baseline comes from the
scaffold doing ~`T`× more weight updates per batch, not from the top-down signal. Activity-level coordination does not move the
wall. *(Recurrence is ~`T`× slower per epoch, so we run fewer epochs here.)*

In [ ]:
class RecFFLayer(nn.Module):
    def __init__(self, dim_below, dim_self, dim_above, lr=0.03):
        super().__init__()
        self.W_up   = nn.Linear(dim_below, dim_self)
        self.W_down = nn.Linear(dim_above, dim_self) if dim_above > 0 else None
        self.relu = nn.ReLU(); self.opt = torch.optim.Adam(self.parameters(), lr=lr); self.threshold = 2.0
    def compute(self, below, above):
        below = below / (below.norm(dim=1, keepdim=True) + 1e-4)
        pre = self.W_up(below)
        if self.W_down is not None and above is not None:
            above = above / (above.norm(dim=1, keepdim=True) + 1e-4)
            pre = pre + self.W_down(above)
        return self.relu(pre)
    def train_step(self, bp, ap, bn, an):
        sp, sn = self.compute(bp, ap), self.compute(bn, an)
        gp, gn = (sp ** 2).mean(1), (sn ** 2).mean(1)
        loss = F.softplus(torch.cat([-(gp - self.threshold), (gn - self.threshold)])).mean()
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        return sp.detach(), sn.detach()

class RecFFNet:
    def __init__(self, dims, use_topdown=True, lr=0.03):
        self.use_topdown = use_topdown; self.layers = []
        L = len(dims) - 1
        for i in range(L):
            da = dims[i+2] if (i + 2 < len(dims) and use_topdown) else 0
            self.layers.append(RecFFLayer(dims[i], dims[i+1], da, lr).to(device))
    def _init(self, B):
        return [torch.zeros(B, ly.W_up.out_features, device=device) for ly in self.layers]
    def train(self, x_pos, x_neg, epochs=30, batch=BATCH, T=8, warmup=2):
        self.T, self.warmup = T, warmup
        n, L = x_pos.shape[0], len(self.layers)
        for _ in range(epochs):
            perm = torch.randperm(n, device=device)
            for i in range(0, n, batch):
                idx = perm[i:i+batch]; xp, xn = x_pos[idx], x_neg[idx]
                sp, sn = self._init(xp.shape[0]), self._init(xn.shape[0])
                for t in range(T):
                    new_p, new_n = [], []                    # synchronous: all read the OLD states
                    for j, layer in enumerate(self.layers):
                        bp = xp if j == 0 else sp[j-1]; ap = sp[j+1] if j < L-1 else None
                        bn = xn if j == 0 else sn[j-1]; an = sn[j+1] if j < L-1 else None
                        if t >= warmup:
                            p, q = layer.train_step(bp, ap, bn, an)
                        else:
                            with torch.no_grad():
                                p, q = layer.compute(bp, ap), layer.compute(bn, an)
                        new_p.append(p); new_n.append(q)
                    sp, sn = new_p, new_n                     # swap only after all layers computed

@torch.no_grad()
def rec_predict(net, x):
    scores = []; L = len(net.layers)
    for label in range(10):
        xl = overlay_label(x, torch.full((x.shape[0],), label, dtype=torch.long, device=device))
        s = [torch.zeros(x.shape[0], ly.W_up.out_features, device=device) for ly in net.layers]; tot = 0
        for t in range(net.T):
            new = []
            for j, layer in enumerate(net.layers):
                below = xl if j == 0 else s[j-1]; above = s[j+1] if j < L-1 else None
                st = layer.compute(below, above); new.append(st)
                if t >= net.warmup: tot = tot + (st ** 2).mean(1)
            s = new
        scores.append(tot.unsqueeze(1))
    return torch.cat(scores, 1).argmax(1)

for flag, name in [(True, "4a. Recurrent top-down ON"), (False, "4b. Recurrent top-down OFF")]:
    net = RecFFNet(DIMS, use_topdown=flag); net.train(pos_train, neg_train, epochs=30)
    RESULTS[name] = eval_acc(net, rec_predict)
    print(f'{name}: {RESULTS[name]:.2f}%')

### Variant 5 — Teacher coordination alignment

**The question.** Earlier variants pointed at *inter-layer coordination* as FF's weak spot: backprop's backward pass lets early
layers adjust for what later layers need, but FF's greedy local rule has no such mechanism. Can we **borrow** that coordination
from a teacher instead of computing it?

**The trick.** Train a backprop MLP (whose layer features *are* globally coordinated). Then train FF as usual, **plus** a small
*local* loss that pulls each FF layer's output direction toward the teacher's matching layer (positives only, direction-only).
The target is fixed and frozen, so **no global gradient ever crosses FF's layers** — FF stays backprop-free, yet the target it
copies secretly carries the teacher's coordination.

**Intuition.** Like studying next to a top student and matching your notes to theirs at every step: you *get up to speed faster*
(faster convergence), but your final exam score ends up the same as learning solo — because what caps you isn't "missing an
answer key for the intermediate steps," it's your own study method (local goodness). A teacher can lend you coordination; it
can't lend you a stronger learning rule.

**Result.** It **never lifts the ceiling.** Gentle alignment (`lam≈1`) gives a small convergence-speed boost and ties plain FF
(~96%); push it hard (`lam≈10`) and it actually *hurts* — forcing FF to mimic features tuned for the teacher's own readout
damages FF's goodness-based separation. Either way, injected coordination is not the wall. *(The teacher also serves as our
backprop **upper-bound** reference, ~98%.)*

In [ ]:
# ---- teacher: backprop MLP (upper bound + coordination source) ----
class TeacherMLP(nn.Module):
    def __init__(self, h=500):
        super().__init__(); self.fc1 = nn.Linear(784, h); self.fc2 = nn.Linear(h, h); self.fc3 = nn.Linear(h, 10)
    def feats(self, x):
        h1 = F.relu(self.fc1(x)); h2 = F.relu(self.fc2(h1)); return h1, h2
    def forward(self, x):
        h1, h2 = self.feats(x); return self.fc3(h2)

teacher = TeacherMLP().to(device); topt = torch.optim.Adam(teacher.parameters(), 1e-3)
for _ in range(10):
    perm = torch.randperm(N, device=device)
    for i in range(0, N, 256):
        idx = perm[i:i+256]
        loss = F.cross_entropy(teacher(plain_train[idx]), label_train[idx])
        topt.zero_grad(); loss.backward(); topt.step()
RESULTS["Teacher (backprop MLP, upper bound)"] = eval_acc(teacher, lambda net, x: net(x).argmax(1))
print(f'teacher: {RESULTS["Teacher (backprop MLP, upper bound)"]:.2f}%')

with torch.no_grad():
    h1T, h2T = teacher.feats(plain_train)
teacher_feats = [h1T.detach(), h2T.detach()]   # FF layer k aligns to teacher_feats[k]

# ---- FF + local alignment to teacher features ----
def align_loss(act, target):                       # 1 - cosine (O(1) magnitude, direction only)
    a = act    / (act.norm(dim=1, keepdim=True)    + 1e-4)
    t = target / (target.norm(dim=1, keepdim=True) + 1e-4)
    return (1 - (a * t).sum(1)).mean()

class AlignFFNet:
    def __init__(self, dims, lr=0.03):
        self.layers = [FFLayer(dims[i], dims[i+1], lr).to(device) for i in range(len(dims)-1)]
    def train(self, x_pos, x_neg, feats, epochs=EPOCHS, batch=BATCH, lam=1.0):
        hp, hn = x_pos, x_neg
        for k, layer in enumerate(self.layers):
            fp = feats[k]; n = hp.shape[0]
            for _ in range(epochs):
                perm = torch.randperm(n, device=device)
                for i in range(0, n, batch):
                    idx = perm[i:i+batch]
                    yp = layer.forward(hp[idx]); yn = layer.forward(hn[idx])
                    gp, gn = (yp ** 2).mean(1), (yn ** 2).mean(1)
                    loss = F.softplus(torch.cat([-(gp - layer.threshold), (gn - layer.threshold)])).mean()
                    loss = loss + lam * align_loss(yp, fp[idx])          # inject coordination (positives only)
                    layer.opt.zero_grad(); loss.backward(); layer.opt.step()
            hp, hn = layer.forward(hp).detach(), layer.forward(hn).detach()

# lam=1.0 -> gentle: ties plain FF (~96). Stronger lam (e.g. 10) over-constrains and HURTS -> coordination is not the wall.
net = AlignFFNet(DIMS); net.train(pos_train, neg_train, teacher_feats, lam=1.0)
RESULTS["5. Teacher coordination align"] = eval_acc(net)
print(f'{RESULTS["5. Teacher coordination align"]:.2f}%')

### Variant 6 — Hard negatives
Use the teacher to pick each image's **most-confusable** wrong label as its negative. The negatives are genuinely harder
(g_neg stays higher), but accuracy **drops**: fixing one hard label per image undersamples the easy rejections it must also
win at test time (coverage collapse).

In [ ]:
with torch.no_grad():
    probs = teacher(plain_train).softmax(1)
    probs[torch.arange(N, device=device), label_train] = -1.0   # mask the true label
    hard_wrong = probs.argmax(1)                                 # most-confusable wrong label
neg_hard = overlay_label(plain_train, hard_wrong)

net = FFNet(DIMS); net.train(pos_train, neg_hard)
RESULTS["6. Hard negatives (teacher-picked)"] = eval_acc(net)
print(f'{RESULTS["6. Hard negatives (teacher-picked)"]:.2f}%')

### Variant 7 — Learned linear readout

**Two separable things.** FF has (a) *how it learns the weights* — local goodness, no backprop — and (b) *how it classifies* —
the **readout**. They are independent. This variant changes only (b); FF's learning is untouched.

**The default readout is a hard-wired, weak classifier.** To classify we overlay each of the 10 labels, run a forward pass, and
pick the label with the highest **summed goodness**. That rule has **no learned parameters** — it's a crude ruler.

**Swap in a learned ruler.** Freeze the FF features and train a single **linear softmax head** on top of them. One subtlety: we
are predicting the label, so the label must *not* be in the input (chicken-and-egg). We feed a **neutral, label-free** input
(first 10 pixels zeroed), take the concatenated normalized hidden activations as the image's representation, and train the head
to map representation → true label. That head uses (shallow) backprop, but it is a *single* linear layer on frozen features —
**not** the deep backprop FF is designed to avoid.

**The readout matters more as the net grows.** On the small `[784,500,500]` net a learned head helps only +0.5 pp, so it *looks*
minor. But the gain grows with capacity — richer features are exactly what the goodness-sum can't exploit (and deep layers
saturate, adding noise to the sum):

| FF net | goodness-sum | learned linear | readout gain |
|---|---:|---:|---:|
| `[784,500,500]` | 96.07% | 96.59% | +0.52 |
| `[784,2000,2000,2000]` | 96.08% | 97.02% | +0.94 |
| `[784,2000,2000,2000,2000]` | 95.85% | **97.12%** | **+1.27** |

At `2000×4` the learned readout reaches **97.1%** — the **first FF number in this study to break the ~96% wall**, within ~0.8 pp
of the backprop teacher.

**What this means (scale-aware decomposition).** On the small net the residual gap is mostly feature quality (readout ~28% /
features ~72%). On a big net the split **flips** (readout ~61% / features ~39%): with a proper classifier, FF's large-net
features are *nearly as good as backprop's*. So the ~96% ceiling was, to a large degree, a **goodness-sum readout** limitation —
not the FF learning rule. FF's features at scale are strong; the naive readout was hiding it.

```
big net [784,2000,2000,2000,2000]:   FF+goodness 95.85 → FF+linear 97.12 → teacher 97.93
                                          └ readout 1.27 (61%) ┘ └ features 0.81 (39%) ┘
```

In [ ]:
@torch.no_grad()
def extract_feats(net, x):
    x = x.clone(); x[:, :10] = 0.0                 # neutral input: no label (we are predicting it)
    feats, h = [], x
    for layer in net.layers:
        h = layer.forward(h)
        feats.append(h / (h.norm(dim=1, keepdim=True) + 1e-4))   # direction only
    return torch.cat(feats, 1)

def readout_compare(dims):
    net = FFNet(dims); net.train(pos_train, neg_train)
    g = eval_acc(net)                                             # (a) goodness-sum readout
    feats_tr = extract_feats(net, plain_train)                   # (b) learned linear readout
    clf = nn.Linear(feats_tr.shape[1], 10).to(device); copt = torch.optim.Adam(clf.parameters(), 1e-3)
    for _ in range(30):
        perm = torch.randperm(feats_tr.shape[0], device=device)
        for i in range(0, feats_tr.shape[0], 256):
            idx = perm[i:i+256]
            loss = F.cross_entropy(clf(feats_tr[idx]), label_train[idx])
            copt.zero_grad(); loss.backward(); copt.step()
    lin = eval_acc(net, lambda n, x: clf(extract_feats(n, x)).argmax(1))
    return g, lin

# The readout gain GROWS with capacity: goodness-sum can't exploit rich features (and deep layers
# saturate, adding noise to the sum) -> a learned head unlocks them and breaks the ~96 wall.
# NOTE: the big nets at 120 epochs are slow; trim this list for a quick run.
print(f"{'FF net':<28}{'goodness':>9}{'linear':>9}{'gain':>8}")
for dims in [[784, 500, 500], [784, 2000, 2000, 2000], [784, 2000, 2000, 2000, 2000]]:
    g, lin = readout_compare(dims)
    print(f"{str(dims):<28}{g:>8.2f}%{lin:>8.2f}%{lin-g:>+7.2f}")
    if dims == DIMS:                                             # standard net feeds the gap decomposition
        RESULTS["7a. FF features + goodness readout"] = g
        RESULTS["7b. FF features + learned linear readout"] = lin

## Overall results

In [ ]:
print("=" * 58)
print("OVERALL RESULTS  (MNIST test accuracy)")
print("=" * 58)
for k, v in RESULTS.items():
    print(f"{v:6.2f}%   {k}")

# gap decomposition (using the readout variant's two numbers vs the teacher)
g  = RESULTS.get("7a. FF features + goodness readout")
lin = RESULTS.get("7b. FF features + learned linear readout")
tea = RESULTS.get("Teacher (backprop MLP, upper bound)")
if g and lin and tea:
    total = tea - g; readout = lin - g; feat = tea - lin
    print("\n" + "=" * 58)
    print("GAP DECOMPOSITION  (FF goodness -> backprop teacher)")
    print("=" * 58)
    print(f"total gap      : {total:.2f} pp")
    print(f"  readout      : {readout:.2f} pp  ({100*readout/total:.0f}%)")
    print(f"  feature qual : {feat:.2f} pp  ({100*feat/total:.0f}%)  <- the real wall")

## Conclusion

Every architectural / data / readout knob was ruled in or out by controlled experiment:

- **Architecture** (width/depth/injection/DenseNet/top-down): all land at ~96% once fully trained (with the naive readout).
- **Training budget**: the real lever for the naive setup (20ep 92% → 120ep 96%), but it too caps at ~96%.
- **Inter-layer coordination** (recurrent top-down, teacher alignment): injecting it does not raise the ceiling.
- **Negative data** (hard / mixed): ~96% either way; hardest-only is worse (coverage collapse).
- **Readout** (learned linear head): **the biggest recoverable factor — and it grows with net size.** On `[784,500,500]` it's
  +0.5 pp, but on `[784,2000,2000,2000,2000]` a learned head reaches **97.1%**, breaking the ~96% wall, within ~0.8 pp of backprop.

**Two-level conclusion (scale-aware):**

1. *With the naive goodness-sum readout*, FF caps at ~96% regardless of architecture — and that ceiling is mostly a **readout**
   limitation, not the learning rule. Goodness-sum can't exploit rich features and is hurt by deep-layer saturation.
2. *The residual gap* (FF+linear 97.1% vs backprop ~97.9%, ~0.8 pp) is **feature quality** — the genuine, small price of local,
   backprop-free learning. It shrinks with scale.

> **Headline:** the ~96% wall is largely the goodness-sum readout. Give FF a big net **and** a learned linear readout and it
> reaches **~97.1%, within ~1 pp of backprop.** FF's features at scale are strong — the naive readout was hiding it.

To close that last ~1 pp you'd need a different *recipe* (Hinton's mask-blended hybrid-image negatives, or convolutional /
local-receptive-field layers), not a bigger architecture. Copying Hinton's 4×2000 *shape* alone does not reproduce his ~98.6%.